# 📖 Capítol 4 - Algorismes i Text

Una seqüència genètica és una cadena (string) formada per caràcters d'un alfabet de quatre lletres: A, T, G, C, anomenats **bases**, que corresponen a les macromolècules de l'**ADN**. Un **gen** és una seqüència ordenada de bases i el **genoma** és la concatenació de tots els gens.

Cada cèl·lula produïda pel cos rep una còpia del genoma, però sovint aquesta còpia és alterada. Les possibles alteracions que es poden produir són, entre d'altres, la substitució d'una base per una altra o la pèrdua d'una base.

### ✍️ Exercici 1. Funció dna 

Fes una funció, anomenada "dna", basada en l'algorisme de Levensthein, que busqui dins d'una seqüència genètica una cadena genètica passada per paràmetre.

Aquesta funció ha de retornar la línia del fitxer on comença la cadena més semblant i la distància entre la cadena d'entrada i la cadena més semblant.

<span style="color:Blue">El càlcul  de la distància d'un patró al *substring* més semblant d'un text es pot fer amb l'algorisme de Levenshtein. L'única diferència és que s'ha d'inicialitzar la primera fila amb zeros i que la distància d'edició serà el valor mínim de l'última fila de la matriu de costos. També has de tenir en compte els costos en la inicialització de la primera columna.</span>


La seqüència genètica que farem servir és la del cromosoma 2 humà (fitxer HUMAN-DNA.txt).

Les primeres línies d'aquest fitxer tenen aquesta forma:

CCCATCTCTTTCTCATTCCTTGGTTGAGAACACGAACTTCAGGACTTGCCTCACACTAGGGCCCATTCTT
TGTTTCCCAGAAAGAAGAGGCTCTCCACACAGAGTCCCATGTACACCAGGCTGTCAACAAACATGAATTG
AATGAAGGAGTGGATGGTTGGGTGGAAGTGATTTAAGAAATCCTAACTGGGGAATTTCACTGGAAACTTA

En programar aquesta funció, cal que tinguis en compte que, en aplicacions bioinformàtiques, els costos de les operacions d'edició són lleugerament diferents dels que hem vist fins ara:

+ Per a un salt o inserció (al patró o al text), el cost és 2
+ Per a una substitució, el cost és 1
+ Quan hi ha correspondència, el cost és 0.

Usa els següents patrons:

In [50]:
# the length of every line in the file is constant
length_of_text = 70
mold_text = [0] * (length_of_text + 1)


def levenstheinsmithwaterman(patro, text, dlt = 2, insr = 2, subs = 1) -> int:
    """
    Aquesta funció implementa l'algorisme de Levensthein amb la variació d'Smith-Waterman. 
    És a dir, inicialitza la primera fila de la matriu a zeros.
    
    Parameters
    ----------
    patro: string
    text: sting
    
    dlt: int (default)
    insr: int (default)
    subs: int (default)
        Costos d'edició
        
    Returns
    -------
    minDistance: int  # Atenció: la distància mínima ja no serà l'extrem de la matriu, cal pensar què serà!
    """
    # not to use a matrix, optimize the algorithm with a rollling-list
    prev_table: list[int] = mold_text.copy()
    # print(prev_table)


    for row in range(1, len(patro) + 1):
        # initialize the curr_table
        curr_table: list[int] = mold_text.copy()
        curr_table[0] = prev_table[0] + dlt

        patro_char: str = patro[row-1]

        for column in range(1, length_of_text + 1):
            text_char: str = text[column-1]
            
            definite_subs = subs if patro_char != text_char else 0
            curr_table[column] = min(
                curr_table[column - 1] + insr, 
                prev_table[column] + dlt, 
                prev_table[column - 1] + definite_subs
            )
        # rolling
        prev_table = curr_table.copy()
        # print(prev_table)
            
    distancia_minima: int = min(prev_table)
    return distancia_minima



def dna(patro, fitxer = 'HUMAN-DNA.txt'):
    """
    Aquesta funció aplica l'algorisme de Levensthein amb la variació d'Smith Waterman 
    sobre una seqüència del dna per trobar diferents patrons.
    Treballa amb fitxers, i fa la cerca a cada línia.

    Parameters
    ----------
    patro: string
    fitxer: string (default)
    
    Returns
    -------
    linia: int
    distanciafinal: int
    """
    linia = 0
    distancia = len(patro) * 2 + 1  # unreachable case

    with open(fitxer, "r") as file:
        for count, line in enumerate(file, start = 1):  # the number of line start with 1
            dist_curr: int = levenstheinsmithwaterman(patro, line)

            if dist_curr < distancia:
                linia = count
                distancia = dist_curr
    
    return (linia,distancia)

In [51]:
assert dna('AGATACATTAGACAATAGAGATGTGGTC') == (32, 11)
assert dna('GTCAGTCTGGCCTTGCCATTGGTGCCACCA') == (352, 11)
assert dna('TACCGAGAAGCTGGATTACAGCATGTACCATCAT') == (233, 13)

Si a més de saber la distància volem saber quins canvis hi ha hagut haurem de modificar els anteriors algorismes per guardar els canvis a cada pas i un cop trobada la distància mínima desfer els passos i anar apuntant els canvis.

Recordem que hi pot haver 4 tipus de canvis

+ I: Insertion
+ D: Deletion
+ S: Substitution
+ C: Coincidence (no hi ha canvis)



### ✍️ Reescriu les anteriors funcions per registrar els canvis i per mostrar-los al final.

In [ ]:
from typing import Any

def levenstheinsmithwaterman(patro, text, dlt = 2, insr = 2, subs = 1):
    """
    Aquesta funció implementa l'algorisme de Levensthein amb la variació de Smith Waterman.
    Guarda a cada casella els canvis que hi ha hagut en una segona matriu de moviments
    
    Parameters
    ----------
    patro: string
    text: sting
    
    dlt: int (default)
    insr: int (default)
    subs: int (default)
        Costos d'edició
        
    Returns
    -------
    inici_text: posició inicial del text més semblant al patró
    final_text: posició final del text més semblant al patró
    distancia_minima: distancia entre el text i el patró
    matriu_moviments: matriu en la que s'indica C,S,D,I segons el moviment fet
    matriu_distancia: matriu amb les distancies
    sequencia_moviment: una llista amb la seqüència de canvis aplicats al patró per arribar al text
    """
    length_patro: int = len(patro)

    value_table, action_table = make_and_initialize_table(patro)
    filling_table(patro, text, value_table, action_table)

    distancia_minima, inici_text, final_text = searching_position(length_patro, value_table[-1])

    sequencia_moviments: tuple[str] = backtrack(action_table, value_table[-1].index(distancia_minima))

    return inici_text,final_text,distancia_minima, sequencia_moviments


def make_and_initialize_table(patro: str) -> tuple[list[list[Any]]]:
    """
    Returns a matrix initialized.
    The size of the matrix is (len(patro) + 1) * (len(text) + 1), corresponds to the algorithm Levenshtein distance,
    the initalization corresponds to the actual, specific case.
    """
    mold_value: list[int] = [0] * (length_of_text + 1)
    value_table: list[list[int]] = [mold_value.copy()]
    mold_action: list[str] = ["C"] * (length_of_text + 1)
    action_table: list[list[str]] = [mold_action.copy()]
    
    # initialize the first column
    for i in range(1, len(patro) + 1):
        value_table.append(mold_value.copy())
        action_table.append(mold_action.copy())
        value_table[i][0] = 2 * i  # one deletion for every char, which costs 2
        action_table[i][0] = "D"

    return value_table, action_table


def filling_table(patro: str, text: str, value_table: list[list[int]], action_table: list[list[str]]) -> None:
    """
    This method fills the value table according to the algorithm of Levenshtein,
    and fills the action table with one of I: Insertion, D: Deletion, S:Substitution, C:Coincidence.
    The cost of them are: 2, 2, 1, 0
    """
    n_row, n_column = len(value_table), len(value_table[0])

    left_cost, left_char = 2, "I"
    up_cost, up_char = 2, "D"

    for row in range(1, n_row):
        for column in range(1, n_column):
            if patro[row - 1] == text[column - 1]:
                diag_cost = 0
                diag_char = "C"
            else:
                diag_cost = 1
                diag_char = "S"

            left_value = value_table[row][column-1] + left_cost
            up_value = value_table[row-1][column] + up_cost

            # suppose the best option is diag
            # if some of the values are the same, the priority is diag_value > up_value > left_value
            best_value = value_table[row - 1][column - 1] + diag_cost
            best_char = diag_char

            if (up_value < best_value):
                best_value = up_value
                best_char = up_char

            if (left_value < best_value):
                best_value = left_value
                best_char = left_char

            value_table[row][column] = best_value
            action_table[row][column] = best_char
    
    # both tables are fully filled
            

def searching_position(length: int, l: list[int]) -> tuple[int]:
    """
    Traverse the l for the minimum integer and it's index at first,
    then, calculate and return the start position and the end position
    """
    minimum: int = min(l)
    index = l.index(minimum) 

    return(minimum, index - length - 1, index - 1)  # position = index - 1


def backtrack(action_table: list[list[int]], column: int) -> list[str]:
    """
    backtrack the index in the action table, and returns the corresponding action
    """
    row: int = len(action_table) - 1
    action_list: list[str] = [action_table[row][column]]
    char_action: dict[str, tuple[int]] = {
        "I" : (0, -1),
        "D" : (-1, 0), 
        "C" : (-1, -1), 
        "S" : (-1, -1)
    }

    action: str = action_table[row][column]
    while (row > 1):  # the row[0] is useless
        i, j = char_action[action]
        row, column = row + i, column + j

        action: str = action_table[row][column]
        action_list.append(action)
    
    action_list.reverse()
    return action_list


def dna(patro, fitxer = 'HUMAN-DNA.txt'):
    """
    Aquesta funció aplica l'algorisme de Levensthein amb la variació de Smith-Waterman sobre una seqüència del dna per trobar diferents patrons.
    
    Parameters
    ----------
    patro: string
    fitxer: string (default)
    
    Returns
    -------
    linia: linia on apareix el patró
    inici_text: posició inicial del text més semblant al patró
    final_text: posició final del text més semblant al patró
    distancia_minima: distancia entre el text i el patró
    se parece que la respuesta no se requere estos dos elementos, los he quitado:
        matriu_moviments: matriu en la que s'indica C,S,D,I segons el moviment fet
        matriu_distancia: matriu amb les distancies
    sequencia_moviment: una llista amb la seqüència de canvis aplicats al patró per arribar al text
    """
    distancia_minima: int = len(patro) * 3  # unreachable value
    linia: int = -1
    with open(fitxer, "r") as file:
        for linia_actual, text in enumerate(file, start = 1):
            inici_text,final_text,distancia_actual, sequencia_moviments = levenstheinsmithwaterman(patro, text)

            if distancia_actual < distancia_minima:
                distancia_minima = distancia_actual
                linia = linia_actual

    print(linia, inici_text, final_text, distancia_minima, sequencia_moviments)
    return (linia,(inici_text,final_text),distancia_minima, sequencia_moviments)

In [61]:
assert dna("TATACAAACGGAGTAGCTGT") == (286, (5, 24), 6, ['C',  'C',  'C',  'C',  'S',  'C',  'S',  'C',  'C',  'S',  'C',  'S',  'S',  'C',  'C',  'C',  'S',  'C',  'C',  'C'])
assert dna("AGGCGTAAGTCTTACGTATA") == (6, (41, 60), 7, ['C',  'S',  'C',  'S',  'S',  'C',  'C',  'C',  'C',  'C',  'C',  'S',  'C',  'S',  'S',  'C',  'S',  'C',  'C',  'C'])
assert dna("AACGGCATAGCCTGCAAGAG") == (434, (41, 60), 5, ['C',  'C',  'S',  'C',  'C',  'C',  'C',  'S',  'C',  'S',  'C',  'C',  'C',  'C',  'C',  'C',  'C',  'S',  'C',  'S'])
assert dna("CTGGTACCAGCTGTATTAGC") == (729,(11, 30), 6, ['C',  'C',  'C',  'C',  'C',  'C',  'C',  'S',  'C',  'S',  'C',  'S',  'S',  'S',  'C',  'C',  'S',  'C',  'C',  'C'])
assert dna("TCGTCATAAACCGCTGTGCC") == (213,(12, 31), 7, ['S',  'C',  'S',  'C',  'C',  'C',  'C',  'C',  'C',  'C',  'C',  'C',  'S',  'S',  'C',  'C',  'S',  'S',  'C',  'S'])


286 8 28 6 ['C', 'S', 'S', 'C', 'C', 'C', 'S', 'C', 'S', 'S', 'S', 'S', 'C', 'S', 'C', 'S', 'C', 'C', 'C', 'S']


AssertionError: 

In [48]:
v, a = make_and_initialize_table("12345")

for row in v:
    print(row)
print()
for row in a:
    print(row)

print()
print("---")
print()

filling_table("12345", "22345223452234555555555555555555555555555555555555555555555555555555555555", v, a)

for row in v:
    print(row)
print()
for row in a:
    print(row)

print()

distancia_minima, inici_text, final_text = searching_position(len("12345"), v[-1])
sequencia_moviments: tuple[str] = backtrack(a, v[-1].index(distancia_minima))

print(sequencia_moviments)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,